<a href="https://colab.research.google.com/github/parthsharma5575/Ai-Ml-GenAi/blob/main/AI_Agents%2CPractical_Implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agents :-


---


This is the most "Advanced" and exciting topic in the world of AI. Everything we've learned so far (GPT, RAG, Prompting) was limited to just giving "Answers". But an AI Agent doesn't just talk — it takes Actions.

To understand it in simple terms:

* LLM: A very big brain that knows everything (The Brain).

* AI Agent: That brain, but with Hands and Legs (Tools) as well.

## 1. Concept: Agentic Workflow (The Manager Analogy)
Imagine you've hired an Assistant.

* If you ask them "What is the price of a laptop?", they won't just guess.

* They will open a Browser (Tool 1), check the price, then add tax using a Calculator (Tool 2), and Email (Tool 3) you the final report.

An Agent has 4 main components:

* Brain (LLM): For thinking and making plans.

* Planning: Breaking the goal down into small steps.

* Memory: Remembering what happened in the previous steps.

* Tools: External APIs, Search, Python Interpreter, etc.

## 2. "Reasoning Loop" (ReAct Framework)

Agents follow a special pattern called ReAct (Reason + Act):

* Thought: "I need to check the weather for the user."

* Action: Call get_weather(city="Delhi").

* Observation: The tool returned "42°C".

* Thought: "It's very hot, I should advise the user to stay cool."

* Final Answer: "It's 42°C in Delhi, please stay indoors."

## 4. Technical Details
Make sure students understand these deeper points:

* Tool Definition:

    Every tool has a Description. The Agent reads that description (via Prompt Engineering) and understands: "Okay, if a calculation is needed, I should use the 'Calculator' tool."

* Stop Sequences:

    The Agent knows when to stop talking and when to wait for the tool's result.

* Agentic Loops (Infinite Loops):

    Sometimes the Agent gets confused and keeps calling the same tool over and over. To prevent this, a max_iterations limit is set.


* Multi-Agent Systems: Nowadays, frameworks like "CrewAI" or "AutoGen" have emerged, where one Agent acts as the "Manager" and another as the "Coder" or "Researcher", and they talk to each other to finish the job.

## 3. Practical Implementation (Google Colab Code)
We should use the LangChain library, which is the most popular for building Agents. Here we will build an Agent that can use a Calculator and Wikipedia.

### Install packages

In [ ]:
!pip install -q langchain langchain-groq langchain-community duckduckgo-search

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("GROQ API KEY:")


GROQ API KEY:··········


In [ ]:
from langchain_groq import ChatGroq
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain.tools import tool
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.prompts import PromptTemplate
import math

In [ ]:
llm = ChatGroq(

               model = 'llama-3.3-70b-versatile',
               temperature = 0
)

In [ ]:
print(llm.model_name)

llama-3.3-70b-versatile


### Create Tools

In [ ]:
@tool

def calculator(expression :str) -> str:
    """
    Useful for solving math calculations.
    Input must be a valid Python math expression.
    Examples: '2 + 2', 'math.sqrt(144)', '15 * 4 / 2'
    """
    try:
      result = eval(expression,{'math':math,'__builtins__':{}})
      return f'result: {result}'

    except Exception as e:
      return f"Error: {str(e)}"


@tool
def get_weather(city:str) ->str:
    """
    Returns current weather for a given city.
    Input should be a city name like 'Delhi', 'Mumbai', 'London'.
    Use this when user asks about weather.
    """

    weather_db = {
        "delhi":    "🌤️  Delhi: 34°C, Partly Cloudy, Humidity 45%",
        "mumbai":   "🌧️  Mumbai: 28°C, Light Rain, Humidity 80%",
        "london":   "☁️  London: 15°C, Overcast, Humidity 70%",
        "new york": "🌞 New York: 22°C, Sunny, Humidity 55%",
        "tokyo":    "🌸 Tokyo: 18°C, Clear, Humidity 60%",
          }

    return weather_db.get(city.lower().strip(),
                          f"no data '{city}. try delhi, mumbai, london, new york")


@tool
def knowledge_base(topic:str) -> str:
    """
    Returns information about AI and tech topics.
    Input should be a topic like 'machine learning', 'LangChain', 'AI agent'.
    Use this when user asks about AI/tech concepts.
    """
    kb = {
        "langchain":        "LangChain is a Python framework for building apps powered by LLMs. It provides tools for chains, agents, memory, and retrieval.",
        "machine learning": "Machine Learning is a subset of AI where models learn patterns from data without being explicitly programmed.",
        "neural network":   "A Neural Network is a system of nodes inspired by the human brain, used for tasks like image recognition and NLP.",
        "llm":              "Large Language Models (LLMs) are trained on massive text data to understand and generate human language. Examples: GPT-4, Gemini, Claude, LLaMA.",
        "ai agent":         "An AI Agent is an LLM that can reason, use tools, and take multi-step actions to achieve a goal autonomously.",
        "groq":             "Groq is a company that provides ultra-fast LLM inference using custom LPU hardware. It offers a free API for models like LLaMA3.",
        "react":            "ReAct (Reason + Act) is a prompting strategy where the agent alternates between reasoning and taking actions to solve tasks.",
    }

    key = topic.lower().strip()
    for k,v in kb.items():
      if k in key or key in k:
        return v
    return f"no data for '{topic}'"


tools = [calculator, get_weather, knowledge_base]

In [ ]:
for i in tools:
  print(i.name)

calculator
get_weather
knowledge_base


### Build Agent


In [ ]:
react_prompt = PromptTemplate.from_template("""You are a helpful AI assistant. Answer the question using the tools available.

You have access to these tools:
{tools}

Use EXACTLY this format:

Question: the input question
Thought: think about what to do
Action: one of [{tool_names}]
Action Input: input to the action
Observation: result of the action
... (repeat Thought/Action/Action Input/Observation as needed)
Thought: I now know the final answer
Final Answer: the final answer to the question

Important rules:
- Always start with Thought:
- Action must be exactly one of the tool names listed
- Always end with Final Answer:

Begin!

Question: {input}
Thought:{agent_scratchpad}""")


In [ ]:
agent = create_react_agent(llm = llm,tools = tools,prompt=react_prompt)

agent_executor = AgentExecutor(

                                agent = agent,
                                tools = tools,
                                verbose = True,
                                max_iterations = 6,
                                handle_parsing_errors = True
                                )




### Cell 6 — Test the Agent

In [ ]:
r = agent_executor.invoke({"input":"What is the square root of 16?"})
print('answer :-',r['output'])



> Entering new AgentExecutor chain...
Thought: To find the square root of 16, I need to use a mathematical operation, so I should use the calculator tool.
Action: calculator
Action Input: math.sqrt(16)result: 4.0Thought: I have successfully calculated the square root of 16 using the calculator tool, so I can now provide the final answer.
Final Answer: The square root of 16 is 4.0

> Finished chain.
answer :- The square root of 16 is 4.0


In [ ]:
r = agent_executor.invoke({"input": "What is the weather in Mumbai?"})
print("\n Answer:", r["output"])



> Entering new AgentExecutor chain...
Thought: The user is asking about the current weather in a specific city, so I need to use the tool that provides weather information.
Action: get_weather
Action Input: Mumbai🌧️  Mumbai: 28°C, Light Rain, Humidity 80%Thought: I have obtained the current weather information for Mumbai, which includes the temperature, precipitation, and humidity, so I can now provide the final answer to the user's question.
Action: NoneInvalid Format: Missing 'Action Input:' after 'Action:'Question: What is the weather in Mumbai?
Thought: The user is asking about the current weather in a specific city, so I need to use the tool that provides weather information.
Action: get_weather
Action Input: Mumbai🌧️  Mumbai: 28°C, Light Rain, Humidity 80%Question: What is the weather in Mumbai?
Thought: The user is asking about the current weather in a specific city, so I need to use the tool that provides weather information.
Action: get_weather
Action Input: Mumbai🌧️  Mumbai

In [ ]:
r = agent_executor.invoke({
    "input": "What is LangChain? Also calculate 18 multiplied by 25. And what's the weather in Delhi?"
})



> Entering new AgentExecutor chain...
Thought: The user has asked three separate questions: one about LangChain, another about a mathematical calculation, and a third about the weather in Delhi. I should first address the question about LangChain.

Action: knowledge_base
Action Input: LangChainLangChain is a Python framework for building apps powered by LLMs. It provides tools for chains, agents, memory, and retrieval.Thought: Now that I have addressed the question about LangChain, I should move on to the mathematical calculation. The user has asked for the result of 18 multiplied by 25.

Action: calculator
Action Input: 18 * 25result: 450Thought: Now that I have the result of the mathematical calculation, I should move on to the question about the weather in Delhi.

Action: get_weather
Action Input: Delhi🌤️  Delhi: 34°C, Partly Cloudy, Humidity 45%Thought: I now know the final answer
Final Answer: LangChain is a Python framework for building apps powered by LLMs. The result of 18 mu

### Add Memory

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.agents import create_react_agent

In [ ]:
memory_prompt = PromptTemplate.from_template("""You are a helpful AI assistant with memory of the conversation.

You have access to these tools:
{tools}

Use EXACTLY this format:

Question: the input question
Thought: think about what to do
Action: one of [{tool_names}]
Action Input: input for the action
Observation: result of the action
... (repeat as needed)
Thought: I now know the final answer
Final Answer: the final answer

Chat History so far:
{chat_history}

Begin!

Question: {input}
Thought:{agent_scratchpad}""")


In [ ]:
memory = ConversationBufferMemory(
    memory_key = 'chat_history',
    return_messages = False
)



In [ ]:
agent_memory = create_react_agent(llm = llm,tools = tools,prompt=memory_prompt)

agent_executor = AgentExecutor(

                                agent = agent_memory,
                                tools = tools,
                                memory = memory,
                                verbose = True,
                                max_iterations = 6,
                                handle_parsing_errors = True
                                )

In [ ]:
r1 = agent_executor.invoke({"input": "Hi! My name is abhishek and I study AI."})
print(":", r1["output"])



> Entering new AgentExecutor chain...
Thought: The user is introducing themselves and mentioning their field of study, which is AI. I can use this opportunity to provide information about AI or ask follow-up questions to engage in a conversation.

Action: knowledge_base
Action Input: AILangChain is a Python framework for building apps powered by LLMs. It provides tools for chains, agents, memory, and retrieval.Question: Hi! My name is abhishek and I study AI.
Thought: The user is introducing themselves and mentioning their field of study, which is AI. I can use this opportunity to provide information about AI or ask follow-up questions to engage in a conversation.
Action: knowledge_base
Action Input: AILangChain is a Python framework for building apps powered by LLMs. It provides tools for chains, agents, memory, and retrieval.Question: Hi! My name is abhishek and I study AI.
Thought: The user is introducing themselves and mentioning their field of study, which is AI. I can use this 